# Marketing Department — A/B Test Analysis & Hypothesis Prioritization

| Field | Details |
| --- | --- |
| **Author** | Marcus Vinícius |
| **Dataset** | `hypotheses_us.csv` · `orders_us.csv` · `visits_us.csv` |
| **Stack** | Python · Pandas · NumPy · SciPy · Seaborn · Matplotlib |
| **Goal** | Prioritize business hypotheses using ICE/RICE frameworks and evaluate an A/B test to determine whether Group B's feature should be permanently adopted |

## Introduction

This project explores business hypotheses alongside the Marketing Department to increase user conversion on the online store. The analysis is divided into two main parts:

1. **Hypothesis Prioritization** — ranking hypotheses using the ICE and RICE scoring frameworks.
2. **A/B Test Analysis** — evaluating cumulative revenue, conversion rate, average order size, and statistical significance between Group A (control) and Group B (treatment).

## Table of Contents

1. [Data Dictionary](#1.-Data-Dictionary)
2. [Setup & Imports](#2.-Setup-&-Imports)
3. [Step 1 — Loading the Data](#3.-Step-1-—-Loading-the-Data)
4. [Step 2 — Data Preparation](#4.-Step-2-—-Data-Preparation)
5. [Step 3 — Hypothesis Prioritization](#5.-Step-3-—-Hypothesis-Prioritization)
6. [Step 4 — A/B Test Analysis](#6.-Step-4-—-A/B-Test-Analysis)
7. [Conclusions](#7.-Conclusions)

## 1. Data Dictionary

Three DataFrames are used throughout this analysis:

**`hypotheses_us.csv`** — Business hypotheses to be prioritized:
| Column | Description |
| --- | --- |
| `Hypothesis` | Text description of the hypothesis |
| `Reach` | Estimated user reach, scale 1–10 |
| `Impact` | Expected impact on users, scale 1–10 |
| `Confidence` | Confidence level in the hypothesis, scale 1–10 |
| `Effort` | Resources required to test the hypothesis, scale 1–10 (higher = more costly) |

**`orders_us.csv`** — Order-level transactional data:
| Column | Description |
| --- | --- |
| `transactionId` | Unique order identifier |
| `visitorId` | Identifier of the user who placed the order |
| `date` | Order date |
| `revenue` | Order revenue (USD) |
| `group` | A/B test group the user belongs to (`A` or `B`) |

**`visits_us.csv`** — Daily site visit data:
| Column | Description |
| --- | --- |
| `date` | Visit date |
| `group` | A/B test group (`A` or `B`) |
| `visits` | Number of visits on that date for that group |

## 2. Setup & Imports

In [ ]:
import pandas as pd
import numpy as np
import math
from scipy import stats as st
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns

# Global plot style
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_context('notebook', font_scale=1.1)

# Color palette
PALETTE = {'A': '#4C72B0', 'B': '#DD8452'}
BLUE = '#4C72B0'
ORANGE = '#DD8452'

def add_value_labels(ax, fmt='{:,.0f}', spacing=4):
    """Annotate bar chart patches with their heights."""
    for rect in ax.patches:
        height = rect.get_height()
        if height == 0:
            continue
        ax.annotate(fmt.format(height),
                    xy=(rect.get_x() + rect.get_width() / 2, height),
                    xytext=(0, spacing), textcoords='offset points',
                    ha='center', va='bottom', fontsize=9,
                    fontweight='bold', color='#374151')

## 3. Step 1 — Loading the Data

In [ ]:
df_hypotheses = pd.read_csv('hypotheses_us.csv', sep=';')
df_orders     = pd.read_csv('orders_us.csv')
df_visits     = pd.read_csv('visits_us.csv')

### Hypotheses DataFrame

In [ ]:
df_hypotheses.head()

In [ ]:
df_hypotheses.info()

### Orders DataFrame

In [ ]:
df_orders.head()

In [ ]:
df_orders.info()

### Visits DataFrame

In [ ]:
df_visits.head()

In [ ]:
df_visits.info()

## 4. Step 2 — Data Preparation

Steps performed:
- Remove duplicate rows
- Check for missing values
- Standardize column names to lowercase
- Convert `date` columns to `datetime`

### Hypotheses DataFrame

In [ ]:
# Remove duplicates
df_hypotheses = df_hypotheses.drop_duplicates()
print('Duplicate rows:', df_hypotheses.duplicated().sum())
print()
print('Missing values:')
print(df_hypotheses.isna().sum())

In [ ]:
# Standardize column names
df_hypotheses.columns = df_hypotheses.columns.str.lower()
df_hypotheses.head()

### Orders DataFrame

In [ ]:
# Remove duplicates
df_orders = df_orders.drop_duplicates()
print('Duplicate rows:', df_orders.duplicated().sum())
print()
print('Missing values:')
print(df_orders.isna().sum())

In [ ]:
# Standardize column names
df_orders.columns = df_orders.columns.str.lower()

# Convert date column to datetime
df_orders['date'] = pd.to_datetime(df_orders['date'])
df_orders.head()

### Visits DataFrame

In [ ]:
# Remove duplicates
df_visits = df_visits.drop_duplicates()
print('Duplicate rows:', df_visits.duplicated().sum())
print()
print('Missing values:')
print(df_visits.isna().sum())

In [ ]:
# Convert date column to datetime
df_visits['date'] = pd.to_datetime(df_visits['date'])
df_visits.head()

## 5. Step 3 — Hypothesis Prioritization

We apply two popular scoring frameworks to rank the nine business hypotheses:

- **ICE** = (Impact × Confidence) / Effort — quick, lightweight scoring.
- **RICE** = (Reach × Impact × Confidence) / Effort — adds user reach for a more nuanced ranking.

In [ ]:
df_hypotheses.head()

### Metric Distribution — Boxplot

In [ ]:
df_boxplot = df_hypotheses.melt(
    id_vars='hypothesis',
    value_vars=['reach', 'impact', 'confidence', 'effort'],
    var_name='metric',
    value_name='value'
)

fig, ax = plt.subplots(figsize=(10, 6))
sns.boxplot(
    x='metric', y='value', data=df_boxplot,
    palette='pastel', ax=ax
)
ax.set_title('Distribution of Scoring Metrics Across Hypotheses', fontsize=14, fontweight='bold', pad=12)
ax.set_xlabel('Metric', fontsize=12)
ax.set_ylabel('Score (1–10)', fontsize=12)
ax.set_xticklabels(['Reach', 'Impact', 'Confidence', 'Effort'])
sns.despine()
plt.tight_layout()
plt.show()

> **Insight:** The `reach` and `confidence` metrics show relatively low dispersion compared to `impact` and `effort`, suggesting stronger agreement across hypotheses on those dimensions. The wider spread in `impact` and `effort` reflects genuine uncertainty about which hypotheses will deliver the most value at the lowest cost.

### ICE Scoring

In [ ]:
df_hypotheses['ICE'] = (
    (df_hypotheses['impact'] * df_hypotheses['confidence'])
    / df_hypotheses['effort']
)
df_hypotheses[['hypothesis', 'ICE']].sort_values(by='ICE', ascending=False).round(2)

### RICE Scoring

In [ ]:
df_hypotheses['RICE'] = (
    (df_hypotheses['reach'] * df_hypotheses['impact'] * df_hypotheses['confidence'])
    / df_hypotheses['effort']
)
df_hypotheses[['hypothesis', 'RICE']].sort_values(by='RICE', ascending=False)

> **Insight:** ICE and RICE produce different rankings because RICE incorporates `reach` — the number of users affected by each hypothesis. This shifts priority away from high-impact/low-reach ideas toward those that affect a broader user base. **Use ICE** for a fast, lightweight evaluation; **use RICE** when breadth of impact matters and data on reach is reliable.

### ICE vs. RICE Score Comparison

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# Shorten hypothesis labels for readability
df_hypotheses['label'] = df_hypotheses['hypothesis'].str[:40] + '...'

# ICE chart
ice_sorted = df_hypotheses.sort_values('ICE', ascending=False)
axes[0].barh(ice_sorted['label'], ice_sorted['ICE'], color=BLUE, edgecolor='white')
axes[0].set_title('ICE Score Ranking', fontsize=13, fontweight='bold')
axes[0].set_xlabel('ICE Score')
axes[0].invert_yaxis()
sns.despine(ax=axes[0])

# RICE chart
rice_sorted = df_hypotheses.sort_values('RICE', ascending=False)
axes[1].barh(rice_sorted['label'], rice_sorted['RICE'], color=ORANGE, edgecolor='white')
axes[1].set_title('RICE Score Ranking', fontsize=13, fontweight='bold')
axes[1].set_xlabel('RICE Score')
axes[1].invert_yaxis()
sns.despine(ax=axes[1])

plt.suptitle('Hypothesis Prioritization: ICE vs. RICE', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

## 6. Step 4 — A/B Test Analysis

We now analyze the A/B test results using the `orders_us` and `visits_us` datasets. The analysis covers:

- Cumulative revenue by group
- Cumulative average order value by group
- Relative difference in average order value
- Daily conversion rate by group
- Relative difference in cumulative conversion rate
- Revenue and orders-per-user distribution
- Outlier detection and filtered statistical tests

In [ ]:
df_orders.head()

In [ ]:
df_visits.head()

### Aggregating Cumulative Orders and Visits

In [ ]:
# Get unique date-group pairs from orders
datesGroups = df_orders[['date', 'group']].drop_duplicates()

# Build cumulative aggregation: for each date and group,
# count all orders/buyers and sum revenue UP TO that date
ordersAggregated = (
    datesGroups.apply(
        lambda x: df_orders[
            np.logical_and(
                df_orders['date'] <= x['date'],
                df_orders['group'] == x['group']
            )
        ].agg({
            'date': 'max',
            'group': 'max',
            'transactionid': pd.Series.nunique,
            'visitorid': pd.Series.nunique,
            'revenue': 'sum'
        }),
        axis=1
    ).sort_values(by=['date', 'group'])
)
ordersAggregated.head()

In [ ]:
# Build cumulative visitor counts
visitorsAggregated = (
    datesGroups.apply(
        lambda x: df_visits[
            np.logical_and(
                df_visits['date'] <= x['date'],
                df_visits['group'] == x['group']
            )
        ].agg({
            'date': 'max',
            'group': 'max',
            'visits': 'sum'
        }),
        axis=1
    ).sort_values(by=['date', 'group'])
)
visitorsAggregated.head()

In [ ]:
# Merge aggregated orders and visits into a single cumulative DataFrame
cumulativeData = ordersAggregated.merge(
    visitorsAggregated, on=['date', 'group']
)
cumulativeData.columns = ['date', 'group', 'orders', 'buyers', 'revenue', 'visitors']
cumulativeData.head()

### Cumulative Revenue — Group A vs. Group B

In [ ]:
cumulativeRevenueA = cumulativeData[cumulativeData['group'] == 'A'][['date', 'revenue', 'orders']]
cumulativeRevenueB = cumulativeData[cumulativeData['group'] == 'B'][['date', 'revenue', 'orders']]

fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(cumulativeRevenueA['date'], cumulativeRevenueA['revenue'],
        label='Group A', color=BLUE, linewidth=2)
ax.plot(cumulativeRevenueB['date'], cumulativeRevenueB['revenue'],
        label='Group B', color=ORANGE, linewidth=2)
ax.set_title('Cumulative Revenue — Group A vs. Group B', fontsize=14, fontweight='bold', pad=12)
ax.set_xlabel('Date', fontsize=12)
ax.set_ylabel('Cumulative Revenue (USD)', fontsize=12)
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x:,.0f}'))
ax.tick_params(axis='x', rotation=45)
ax.legend(fontsize=11)
sns.despine()
plt.tight_layout()
plt.show()

> **Insight:** Group A's revenue grew steadily throughout the period. Group B experienced a sharp spike around 2019-08-17 — likely driven by a few very high-value orders (outliers) — after which it stabilized and ended the period ahead of Group A in total cumulative revenue.

### Cumulative Average Order Value — Group A vs. Group B

In [ ]:
fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(cumulativeRevenueA['date'],
        cumulativeRevenueA['revenue'] / cumulativeRevenueA['orders'],
        label='Group A', color=BLUE, linewidth=2)
ax.plot(cumulativeRevenueB['date'],
        cumulativeRevenueB['revenue'] / cumulativeRevenueB['orders'],
        label='Group B', color=ORANGE, linewidth=2)
ax.set_title('Cumulative Average Order Value — Group A vs. Group B',
             fontsize=14, fontweight='bold', pad=12)
ax.set_xlabel('Date', fontsize=12)
ax.set_ylabel('Average Order Value (USD)', fontsize=12)
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x:,.0f}'))
ax.tick_params(axis='x', rotation=45)
ax.legend(fontsize=11)
sns.despine()
plt.tight_layout()
plt.show()

> **Insight:** The average order value chart mirrors the revenue pattern — the same spike appears around 2019-08-17, confirming the presence of unusually large orders on that date. Outside of that spike, both groups show a broadly similar and relatively stable average order value, with some fluctuations.

### Relative Difference in Average Order Value — Group B vs. Group A

In [ ]:
mergedCumulativeRevenue = cumulativeRevenueA.merge(
    cumulativeRevenueB, on='date', how='left', suffixes=['A', 'B']
)

relative_diff = (
    (mergedCumulativeRevenue['revenueB'] / mergedCumulativeRevenue['ordersB'])
    / (mergedCumulativeRevenue['revenueA'] / mergedCumulativeRevenue['ordersA'])
    - 1
)

fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(mergedCumulativeRevenue['date'], relative_diff, color=BLUE, linewidth=2)
ax.axhline(y=0, color='black', linestyle='--', linewidth=1, label='No difference')
ax.fill_between(mergedCumulativeRevenue['date'], relative_diff, 0,
                where=(relative_diff > 0), alpha=0.15, color=ORANGE, label='B > A')
ax.fill_between(mergedCumulativeRevenue['date'], relative_diff, 0,
                where=(relative_diff < 0), alpha=0.15, color=BLUE, label='A > B')
ax.set_title('Relative Difference in Average Order Value: Group B vs. Group A',
             fontsize=14, fontweight='bold', pad=12)
ax.set_xlabel('Date', fontsize=12)
ax.set_ylabel('Relative Difference', fontsize=12)
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x:.0%}'))
ax.tick_params(axis='x', rotation=45)
ax.legend(fontsize=11)
sns.despine()
plt.tight_layout()
plt.show()

> **Insight:** The relative difference chart confirms that outlier orders are skewing the average order value metric significantly. This motivates filtering out anomalous users before drawing final conclusions about order value differences between the two groups.

### Daily Conversion Rate — Group A vs. Group B

In [ ]:
visits_per_day  = df_visits.groupby(['date', 'group']).agg({'visits': 'sum'})
orders_per_day  = df_orders.groupby(['date', 'group']).agg(
    {'transactionid': 'nunique'}
).reset_index()

dailyConversion = pd.merge(visits_per_day, orders_per_day,
                           on=['date', 'group'], how='left')
dailyConversion['transactionid'] = dailyConversion['transactionid'].fillna(0)
dailyConversion['conversion'] = (
    dailyConversion['transactionid'] / dailyConversion['visits']
)

cumulativeDataA = dailyConversion[dailyConversion['group'] == 'A']
cumulativeDataB = dailyConversion[dailyConversion['group'] == 'B']

fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(cumulativeDataA.index.get_level_values('date'),
        cumulativeDataA['conversion'], label='Group A', color=BLUE, linewidth=2)
ax.plot(cumulativeDataB.index.get_level_values('date'),
        cumulativeDataB['conversion'], label='Group B', color=ORANGE, linewidth=2)
ax.set_title('Daily Conversion Rate — Group A vs. Group B',
             fontsize=14, fontweight='bold', pad=12)
ax.set_xlabel('Date', fontsize=12)
ax.set_ylabel('Conversion Rate', fontsize=12)
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x:.1%}'))
ax.tick_params(axis='x', rotation=45)
ax.legend(fontsize=11)
sns.despine()
plt.tight_layout()
plt.show()

> **Insight:** The daily conversion rates for both groups follow a broadly symmetric pattern. A notable spike is visible around 2019-08-09, which is consistent with outlier activity detected elsewhere in the data.

### Relative Difference in Cumulative Conversion Rate — Group B vs. Group A

In [ ]:
mergedCumulativeConversions = (
    cumulativeDataA[['conversion']].rename(columns={'conversion': 'conversionA'})
    .merge(
        cumulativeDataB[['conversion']].rename(columns={'conversion': 'conversionB'}),
        left_index=True, right_index=True, how='left'
    )
)

rel_conv_diff = (
    mergedCumulativeConversions['conversionB']
    / mergedCumulativeConversions['conversionA']
    - 1
)

dates_index = mergedCumulativeConversions.index.get_level_values('date')

fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(dates_index, rel_conv_diff, color=BLUE, linewidth=2)
ax.axhline(y=0,    color='black', linestyle='--', linewidth=1, label='No difference')
ax.axhline(y=0.15, color='grey',  linestyle='--', linewidth=1, label='+15% threshold')
ax.fill_between(dates_index, rel_conv_diff, 0,
                where=(rel_conv_diff > 0), alpha=0.15, color=ORANGE, label='B > A')
ax.set_title('Relative Difference in Cumulative Conversion Rate: Group B vs. Group A',
             fontsize=14, fontweight='bold', pad=12)
ax.set_xlabel('Date', fontsize=12)
ax.set_ylabel('Relative Difference', fontsize=12)
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x:.0%}'))
ax.set_ylim(-0.5, 0.5)
ax.tick_params(axis='x', rotation=45)
ax.legend(fontsize=11)
sns.despine()
plt.tight_layout()
plt.show()

> **Insight:** Group B consistently leads Group A in cumulative conversion rate. After a brief period below the baseline, Group B's conversion advantage stabilized above 15% relative to Group A — a meaningful and sustained lift throughout the test period.

### Revenue Distribution — Histogram & Scatter Plot

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Histogram
axes[0].hist(df_orders['revenue'], bins=70, color=BLUE, edgecolor='white', alpha=0.85)
axes[0].set_title('Revenue Distribution (Histogram)', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Order Revenue (USD)', fontsize=11)
axes[0].set_ylabel('Number of Orders', fontsize=11)
axes[0].xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x:,.0f}'))
sns.despine(ax=axes[0])

# Scatter
x_values = pd.Series(range(len(df_orders['revenue'])))
axes[1].scatter(x_values, df_orders['revenue'], color=ORANGE, alpha=0.5, s=15)
axes[1].set_title('Revenue Distribution (Scatter)', fontsize=13, fontweight='bold')
axes[1].set_xlabel('Order Index', fontsize=11)
axes[1].set_ylabel('Order Revenue (USD)', fontsize=11)
axes[1].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x:,.0f}'))
sns.despine(ax=axes[1])

plt.suptitle('Order Revenue Distribution — All Groups Combined',
             fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

> **Insight:** The vast majority of orders are low-value, with a long right tail driven by a small number of unusually expensive orders. These high-revenue outliers can distort group comparisons and must be accounted for in statistical testing.

### Revenue Percentiles

In [ ]:
p90, p95, p99 = np.percentile(df_orders['revenue'], [90, 95, 99])
print(f'90th percentile: ${p90:,.2f}')
print(f'95th percentile: ${p95:,.2f}')
print(f'99th percentile: ${p99:,.2f}')

> **Insight:** Less than 1% of orders exceeded $900. This confirms that a very small subset of transactions is responsible for the bulk of extreme revenue values observed in the distribution.

### Orders Per User — Distribution

In [ ]:
ordersByUsers = (
    df_orders.drop(['group', 'revenue', 'date'], axis=1)
    .groupby('visitorid', as_index=False)
    .agg({'transactionid': pd.Series.nunique})
)
ordersByUsers.columns = ['user_id', 'orders']

print('Top users by order count:')
print(ordersByUsers.sort_values('orders', ascending=False).head())

> **Insight:** A handful of users placed 8–11 orders during the study period, which is far above the norm. These users are statistical outliers and may skew conversion and revenue metrics.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Histogram
axes[0].hist(ordersByUsers['orders'], bins=20, color=BLUE, edgecolor='white', alpha=0.85)
axes[0].set_title('Orders per User (Histogram)', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Number of Orders', fontsize=11)
axes[0].set_ylabel('Number of Users', fontsize=11)
sns.despine(ax=axes[0])

# Scatter
x_vals = pd.Series(range(len(ordersByUsers['orders'])))
axes[1].scatter(x_vals, ordersByUsers['orders'], color=ORANGE, alpha=0.5, s=15)
axes[1].set_title('Orders per User (Scatter)', fontsize=13, fontweight='bold')
axes[1].set_xlabel('User Index', fontsize=11)
axes[1].set_ylabel('Number of Orders', fontsize=11)
sns.despine(ax=axes[1])

plt.suptitle('Orders per User Distribution',
             fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

### Orders-per-User Percentiles

In [ ]:
p90, p95, p99 = np.percentile(ordersByUsers['orders'], [90, 95, 99])
print(f'90th percentile: {p90:.0f} orders')
print(f'95th percentile: {p95:.0f} orders')
print(f'99th percentile: {p99:.0f} orders')

> **Insight:** 90% of users placed 1–2 orders. Less than 1% placed more than 4 orders. Users exceeding this threshold are classified as anomalous for filtering purposes.

### Statistical Significance — Conversion Rate (All Users)

In [ ]:
ordersByUsersA = (
    df_orders[df_orders['group'] == 'A']
    .groupby('visitorid', as_index=False)
    .agg({'transactionid': pd.Series.nunique})
)
ordersByUsersA.columns = ['user_id', 'orders']

ordersByUsersB = (
    df_orders[df_orders['group'] == 'B']
    .groupby('visitorid', as_index=False)
    .agg({'transactionid': pd.Series.nunique})
)
ordersByUsersB.columns = ['user_id', 'orders']

# Build full samples including non-purchasing visitors (0 orders)
sampleA = pd.concat([
    ordersByUsersA['orders'],
    pd.Series(0,
              index=np.arange(
                  df_visits[df_visits['group'] == 'A']['visits'].sum()
                  - len(ordersByUsersA)
              ),
              name='orders')
], axis=0)

sampleB = pd.concat([
    ordersByUsersB['orders'],
    pd.Series(0,
              index=np.arange(
                  df_visits[df_visits['group'] == 'B']['visits'].sum()
                  - len(ordersByUsersB)
              ),
              name='orders')
], axis=0)

p_value  = st.mannwhitneyu(sampleA, sampleB)[1]
rel_diff = sampleB.mean() / sampleA.mean() - 1

print(f'Mann-Whitney p-value (conversion): {p_value:.3f}')
print(f'Relative difference in mean conversion (B vs A): {rel_diff:.1%}')

> **Insight:** At a 5% significance level (p = 0.017 < 0.05), the difference in conversion rate between groups is **statistically significant**. Group B's mean conversion rate is approximately **13.8% higher** than Group A's.

### Statistical Significance — Average Order Value (All Users)

In [ ]:
p_value  = st.mannwhitneyu(
    df_orders[df_orders['group'] == 'A']['revenue'],
    df_orders[df_orders['group'] == 'B']['revenue']
)[1]
rel_diff = (
    df_orders[df_orders['group'] == 'B']['revenue'].mean()
    / df_orders[df_orders['group'] == 'A']['revenue'].mean()
    - 1
)

print(f'Mann-Whitney p-value (average order value): {p_value:.3f}')
print(f'Relative difference in mean revenue (B vs A): {rel_diff:.1%}')

> **Insight:** At a 5% significance level (p = 0.692 >> 0.05), there is **no statistically significant** difference in average order value between the groups. Despite Group B appearing 25% higher in raw means, this difference is likely driven by outliers. The null hypothesis is not rejected.

### Anomalous User Detection

Users are flagged as anomalous if they meet either of the following criteria:
- Placed **more than 4 orders** (above the 99th percentile)
- Placed **at least one order exceeding $900** (above the 99th percentile of revenue)

In [ ]:
users_many_orders     = pd.concat([
    ordersByUsersA[ordersByUsersA['orders'] > 4]['user_id'],
    ordersByUsersB[ordersByUsersB['orders'] > 4]['user_id']
], axis=0)

users_expensive_orders = df_orders[df_orders['revenue'] > 900]['visitorid']

abnormalUsers = (
    pd.concat([users_many_orders, users_expensive_orders], axis=0)
    .drop_duplicates()
    .sort_values()
)

print(f'Number of anomalous users identified: {len(abnormalUsers)}')
print(abnormalUsers.values)

> **Insight:** 15 anomalous users were identified — those who placed more than 4 orders or had at least one order exceeding $900. These users represent less than 1% of the total sample and will be excluded in the filtered statistical tests below.

### Filtered Statistical Significance — Conversion Rate (Anomalous Users Excluded)

In [ ]:
sampleAFiltered = pd.concat([
    ordersByUsersA[
        ~ordersByUsersA['user_id'].isin(abnormalUsers)
    ]['orders'],
    pd.Series(0,
              index=np.arange(
                  df_visits[df_visits['group'] == 'A']['visits'].sum()
                  - len(ordersByUsersA)
              ),
              name='orders')
], axis=0)

sampleBFiltered = pd.concat([
    ordersByUsersB[
        ~ordersByUsersB['user_id'].isin(abnormalUsers)
    ]['orders'],
    pd.Series(0,
              index=np.arange(
                  df_visits[df_visits['group'] == 'B']['visits'].sum()
                  - len(ordersByUsersB)
              ),
              name='orders')
], axis=0)

p_value  = st.mannwhitneyu(sampleAFiltered, sampleBFiltered)[1]
rel_diff = sampleBFiltered.mean() / sampleAFiltered.mean() - 1

print(f'Mann-Whitney p-value (conversion, filtered): {p_value:.3f}')
print(f'Relative difference in mean conversion (B vs A, filtered): {rel_diff:.1%}')

> **Insight:** After removing anomalous users, the conversion rate difference remains **statistically significant** (p = 0.014 < 0.05). Group B's conversion rate is **~15.3% higher** than Group A's — slightly larger than the unfiltered result, suggesting outliers were mildly suppressing the signal.

### Filtered Statistical Significance — Average Order Value (Anomalous Users Excluded)

In [ ]:
revenue_A_filtered = df_orders[
    (df_orders['group'] == 'A') &
    (~df_orders['visitorid'].isin(abnormalUsers))
]['revenue']

revenue_B_filtered = df_orders[
    (df_orders['group'] == 'B') &
    (~df_orders['visitorid'].isin(abnormalUsers))
]['revenue']

p_value  = st.mannwhitneyu(revenue_A_filtered, revenue_B_filtered)[1]
rel_diff = revenue_B_filtered.mean() / revenue_A_filtered.mean() - 1

print(f'Mann-Whitney p-value (avg order value, filtered): {p_value:.3f}')
print(f'Relative difference in mean revenue (B vs A, filtered): {rel_diff:.1%}')

> **Insight:** After filtering anomalous users, the difference in average order value is **not statistically significant** (p = 0.819 >> 0.05), and the relative revenue difference drops to just **-0.6%** — essentially zero. This confirms that the raw revenue difference was entirely driven by outliers, not by a genuine treatment effect.

## 7. Conclusions

### Summary of Key Findings

| Metric | Raw Result | Filtered Result (outliers removed) |
| --- | --- | --- |
| Conversion rate significance (p-value) | 0.017 ✓ | 0.014 ✓ |
| Conversion rate lift (B vs A) | +13.8% | +15.3% |
| Avg order value significance (p-value) | 0.692 ✗ | 0.819 ✗ |
| Avg order value lift (B vs A) | +25.2% | -0.6% |
| Anomalous users identified | — | 15 users (<1% of sample) |

### Key Takeaways

1. **The A/B test was a success for conversion rate.** Group B's conversion rate is statistically significantly higher than Group A's, both before and after removing outliers. The true lift is approximately **+15%**.

2. **The apparent revenue lift is entirely driven by outliers.** Once anomalous users are excluded, the average order value difference between groups falls to near zero and is not statistically significant. The two groups perform equivalently in terms of order value.

3. **15 anomalous users** — those placing more than 4 orders or spending over $900 in a single order — disproportionately influenced raw revenue metrics. These represent less than 1% of the total sample.

4. **Hypothesis prioritization revealed distinct rankings under ICE vs. RICE.** RICE is more appropriate when reach data is reliable, as it surfaces high-impact hypotheses that benefit a broad user base. The top-ranked RICE hypothesis ('Add a subscription form to all main pages') differs substantially from the top ICE hypothesis ('Launch a promotion with user discounts').

5. **Recommendation:** Stop the A/B test and implement Group B's feature permanently. The conversion rate improvement is real, statistically robust, and sustained throughout the test period. Proceed to test the next highest-priority hypothesis from the RICE ranking.